# 07 – Final Model Selection and Deployment Package
**IT3091 Machine Learning · Group 2026-DS-02 · Bank Marketing**

Brings all evidence together, selects the final model and operating rule, verifies the exact final configuration on the test set,
and produces the model the bank would use.

| Section | Content |
|---|---|
| 1 | Setup |
| 2 | Selection matrix: all evidence in one table |
| 3 | Final configuration: Random Forest without `age` and `marital` |
| 4 | Operating thresholds: **Capacity 20% (main)** and **Max F1 (alternative)**, chosen on training out-of-fold predictions |
| 5 | Final test-set verification (overall, per year, per group) |
| 6 | Refit on all data and save the deployment package |
| 7 | Demo: scoring new clients into a ranked call list |
| 8 | Summary |

⏱ Runtime ≈ 2–4 minutes.

## 1. Setup

In [1]:
import sys, json, time, warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import shap
import sklearn
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config
from src import preprocessing as prep
from src import evaluation as ev

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

MODEL_ORDER = ["Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"]
X_train, X_test, y_train, y_test = prep.load_split()
R = config.RESULTS_DIR
print(f"Train {X_train.shape} | Test {X_test.shape}")

Train (32940, 19) | Test (8236, 19)


## 2. Selection matrix
All numbers are read from the saved results of earlier notebooks (`04_modelling`, `05_evaluation`), so nothing is recomputed or re-tuned here.

In [2]:
test = pd.read_csv(R / "test_results.csv", index_col=0)
boot = pd.read_csv(R / "bootstrap_ci.csv", index_col=0)
cv = pd.read_csv(R / "cv_comparison.csv", index_col=0)
overfit = pd.read_csv(R / "overfitting_check.csv", index_col=0)
per_year = pd.read_csv(R / "per_year_test.csv", index_col=0)

def file_mb(name):
    return (config.MODELS_DIR / f"{name.lower().replace(' ', '_')}.joblib").stat().st_size / 1e6

matrix = pd.DataFrame({
    "CV PR-AUC": cv.loc[MODEL_ORDER, "pr_auc_mean"].round(3),
    "Test PR-AUC": test.loc[MODEL_ORDER, "pr_auc"].round(3),
    "Test 95% CI": [f"[{boot.loc[m, '95% CI low']:.3f}, {boot.loc[m, '95% CI high']:.3f}]" for m in MODEL_ORDER],
    "Gain vs LR (test)": [np.nan] + [round(boot.loc[f"{m} - Logistic Regression", "PR-AUC (test)"], 3)
                                     for m in MODEL_ORDER[1:]],
    "Test ROC-AUC": test.loc[MODEL_ORDER, "roc_auc"].round(3),
    "Test top-20% capture": test.loc[MODEL_ORDER, "top20_capture"].round(3),
    "Worst within-year ROC-AUC": [per_year[f"{m} ROC-AUC"].min() for m in MODEL_ORDER],
    "Train-CV gap %": overfit.loc[MODEL_ORDER, "gap_%_of_cv"],
    "Fit time / fold (s)": cv.loc[MODEL_ORDER, "fit_time_s"].round(1),
    "Model size (MB)": [round(file_mb(m), 2) for m in MODEL_ORDER],
    "Interpretability": ["High (odds ratios)", "High (rules)", "Medium (SHAP)", "Medium (SHAP)"],
}, index=MODEL_ORDER)
matrix.T

,Logistic Regression,Decision Tree,Random Forest,XGBoost
CV PR-AUC,0.438,0.443,0.47,0.467
Test PR-AUC,0.446,0.465,0.496,0.484
Test 95% CI,"[0.413, 0.482]","[0.431, 0.499]","[0.461, 0.530]","[0.449, 0.520]"
Gain vs LR (test),NaN,0.018,0.049,0.038
Test ROC-AUC,0.797,0.792,0.814,0.814
Test top-20% capture,0.643,0.642,0.671,0.665
Worst within-year ROC-AUC,0.56,0.537,0.588,0.589
Train-CV gap %,0.5,7.9,25.5,14.0
Fit time / fold (s),0.7,0.6,12.9,5.0
Model size (MB),0.01,0.04,21.09,1.5


**Decision (D27): Random Forest.**
- Best on every **performance** view: test PR-AUC (0.496), top-20% capture, and within-year ranking in all three years.
- Significantly better than the baseline (+0.049 PR-AUC, CI well above 0) and than XGBoost (+0.011, CI [0.003, 0.019]).
- Its drawbacks (largest train–CV gap, slower training, larger file) do not matter for a bank scoring a few thousand clients per
  campaign, and the gap did **not** hurt generalisation (test ≥ CV).
- **XGBoost is the documented fallback** if training speed / model size become constraints (≈ 1 point lower PR-AUC).
- **Operating rule:** *Capacity 20%* (main), *Max F1* (alternative).

## 3. Final configuration
Random Forest with the tuned hyperparameters from `04_modelling`, **without `age` and `marital`** (decision D26: removing them cost
nothing, PR-AUC 0.4959 vs 0.4955, and follows the data-minimisation principle). `housing` and `loan` stay excluded (D15).

In [3]:
rf_tuned = joblib.load(config.MODELS_DIR / "random_forest.joblib")
FINAL_DROP = prep.DROPPED_FEATURES + ["age", "marital"]
final_cats, final_nums = prep.feature_lists("tree", drop=FINAL_DROP)

def make_final_model():
    return Pipeline([("prep", prep.build_preprocessor("tree", drop=FINAL_DROP)),
                     ("model", clone(rf_tuned.named_steps["model"]))])

rf_params = {k: v for k, v in rf_tuned.named_steps["model"].get_params().items()
             if k in ["n_estimators", "max_depth", "min_samples_leaf", "max_features", "class_weight"]}
print("Hyperparameters:", rf_params)
print("Categorical inputs:", final_cats)
print("Numeric inputs:   ", final_nums)
print("Excluded:         ", FINAL_DROP + config.LEAKAGE_FEATURES + ["pdays (replaced by contacted_before)"])

Hyperparameters: {'class_weight': 'balanced', 'max_depth': 12, 'max_features': 0.3, 'min_samples_leaf': 10, 'n_estimators': 300}
Categorical inputs: ['job', 'education', 'default', 'contact', 'month', 'day_of_week', 'poutcome']
Numeric inputs:    ['campaign', 'previous', 'contacted_before', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Excluded:          ['housing', 'loan', 'age', 'marital', 'duration', 'pdays (replaced by contacted_before)']


## 4. Operating thresholds (training out-of-fold predictions)
Same leakage-free method as `05_evaluation` (D22), now for the exact final configuration:
- **Capacity 20% (main rule):** threshold = score above which the top 20% of clients fall. *In practice: "call the top 20% of the list".*
- **Max F1 (alternative):** threshold with the best balance of precision and recall. *Use when calls are expensive or staff is limited.*

In [4]:
start = time.time()
oof = cross_val_predict(make_final_model(), X_train, y_train, cv=ev.CV, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof)
f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
THRESHOLDS = {
    "Capacity 20% (main)": float(np.quantile(oof, 0.80)),
    "Max F1 (alternative)": float(thr[np.argmax(f1)]),
}
print(f"OOF predictions in {time.time() - start:.0f}s | OOF PR-AUC {average_precision_score(y_train, oof):.4f}")
pd.Series(THRESHOLDS, name="threshold").round(4)

OOF predictions in 27s | OOF PR-AUC 0.4658


Capacity 20% (main)     0.4472
Max F1 (alternative)    0.6377
Name: threshold, dtype: float64

## 5. Final test-set verification
The final configuration is trained on the training set only and evaluated **once** on the test set, at both operating rules.

In [5]:
final_train = make_final_model().fit(X_train, y_train)
scores = final_train.predict_proba(X_test)[:, 1]
rf_all_scores = rf_tuned.predict_proba(X_test)[:, 1]

def call_outcome(y, p, t):
    y = np.asarray(y)
    called = p >= t
    hits = int((called & (y == 1)).sum())
    return {"threshold": t, "% called": 100 * called.mean(), "calls": int(called.sum()),
            "subscribers found": hits, "missed": int(y.sum() - hits), "wasted calls": int(called.sum() - hits),
            "recall": hits / y.sum(), "conversion (precision)": hits / max(called.sum(), 1),
            "F1": 2 * hits / (called.sum() + y.sum())}

overall = pd.DataFrame({
    "Final RF (no age/marital)": ev.score_predictions(y_test, scores),
    "Tuned RF (all features, 05_evaluation)": ev.score_predictions(y_test, rf_all_scores),
}).T[["pr_auc", "roc_auc", "top20_capture", "balanced_acc"]]
print("Threshold-free performance on the test set:")
display(overall.round(4))

ops = pd.DataFrame({rule: call_outcome(y_test, scores, t) for rule, t in THRESHOLDS.items()}).T
ops.loc["Call everyone"] = pd.Series(call_outcome(y_test, np.ones(len(y_test)), 0.5))
ops["conversion vs calling everyone (x)"] = ops["conversion (precision)"] / ops.loc["Call everyone", "conversion (precision)"]
print("Operating rules on the test set:")
ops.round(3)

Threshold-free performance on the test set:


,pr_auc,roc_auc,top20_capture,balanced_acc
Final RF (no age/marital),0.4959,0.8140,0.6703,0.7621
"Tuned RF (all features, 05_evaluation)",0.4955,0.8137,0.6713,0.7651


Operating rules on the test set:


,threshold,% called,calls,subscribers found,missed,wasted calls,recall,conversion (precision),F1,conversion vs calling everyone (x)
Capacity 20% (main),0.447,20.046,1651.0,622.0,306.0,1029.0,0.670,0.377,0.482,3.344
Max F1 (alternative),0.638,13.660,1125.0,536.0,392.0,589.0,0.578,0.476,0.522,4.228
Call everyone,0.500,100.000,8236.0,928.0,0.0,7308.0,1.000,0.113,0.203,1.000


In [6]:
clean, _, Xte_pos, _, _ = prep.recreate_split_with_positions()
pd.testing.assert_frame_equal(Xte_pos.reset_index(drop=True), X_test, check_dtype=False)
year = prep.reconstruct_year(clean["month"]).loc[Xte_pos.index].to_numpy()
y_arr = y_test.to_numpy()
rows = []
for yr in sorted(np.unique(year)):
    m = year == yr
    rows.append({"year": yr, "test rows": int(m.sum()), "yes-rate %": round(100 * y_arr[m].mean(), 1),
                 "ROC-AUC": round(roc_auc_score(y_arr[m], scores[m]), 3),
                 "PR-AUC lift": round(average_precision_score(y_arr[m], scores[m]) / y_arr[m].mean(), 2)})
final_per_year = pd.DataFrame(rows).set_index("year")
print("Within-year ranking (time-drift check):")
display(final_per_year)

t_main = THRESHOLDS["Capacity 20% (main)"]
called = scores >= t_main
def groups(series):
    rows = []
    for g in pd.Series(series).dropna().unique():
        m = np.asarray(series == g)
        yy, cc = y_arr[m], called[m]
        rows.append({"group": g, "clients": int(m.sum()), "actual yes %": round(100 * yy.mean(), 1),
                     "selected %": round(100 * cc.mean(), 1),
                     "recall %": round(100 * (cc & (yy == 1)).sum() / max(yy.sum(), 1), 1),
                     "precision %": round(100 * (cc & (yy == 1)).sum() / max(cc.sum(), 1), 1)})
    return pd.DataFrame(rows).set_index("group")
age_group = pd.cut(X_test["age"], bins=prep.AGE_BINS, labels=prep.AGE_LABELS)
print("Group outcomes at the main rule (age/marital are NOT model inputs; used only for monitoring):")
final_fair_age = groups(age_group).reindex(prep.AGE_LABELS)
final_fair_marital = groups(X_test["marital"]).sort_values("clients", ascending=False)
display(final_fair_age)
display(final_fair_marital)

Within-year ranking (time-drift check):


,test rows,yes-rate %,ROC-AUC,PR-AUC lift
year,,,,
2008,5566,4.5,0.592,2.30
2009,2277,20.6,0.788,2.43
2010,393,53.2,0.745,1.45


Group outcomes at the main rule (age/marital are NOT model inputs; used only for monitoring):


,clients,actual yes %,selected %,recall %,precision %
group,,,,,
17-25,328,18.9,43.6,87.1,37.8
26-30,1106,13.8,25.4,69.9,38.1
31-40,3312,10.9,17.1,62.2,39.5
41-50,2021,7.5,14.0,54.3,29.1
51-60,1302,9.4,16.6,62.6,35.6
61-70,92,41.3,98.9,100.0,41.8
71+,75,54.7,94.7,97.6,56.3


,clients,actual yes %,selected %,recall %,precision %
group,,,,,
married,5006,9.8,17.6,65.1,36.4
single,2307,14.3,26.8,73.6,39.3
divorced,911,11.4,16.2,55.8,39.2
unknown,12,8.3,8.3,0.0,0.0


## 6. Refit on all data and save the deployment package
Evaluation is complete, so the final model is **refitted on all 41,176 clients** (train + test). Using all data, especially the most
recent clients, matters because of time drift (Steps 4 and 6). The performance reported for this model remains the **test-set estimate
from Section 5**; the refitted model cannot be re-evaluated on data it has seen.

For the Capacity rule, deployment uses the **ranking** ("call the top 20% of the current list"), which stays valid when the score distribution
of a new list differs. The numeric thresholds are stored for reference.

In [7]:
X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)
start = time.time()
final_model = make_final_model().fit(X_all, y_all)
print(f"Final model refitted on {len(X_all):,} clients in {time.time() - start:.0f}s")

joblib.dump(final_model, config.MODELS_DIR / "final_model.joblib")
final_config = {
    "model": "RandomForestClassifier (scikit-learn Pipeline)",
    "created": str(date.today()),
    "trained_on_rows": int(len(X_all)),
    "dataset_sha256": config.RAW_FILE_SHA256,
    "hyperparameters": {k: (v.item() if isinstance(v, np.generic) else v) for k, v in rf_params.items()},
    "categorical_inputs": final_cats,
    "numeric_inputs": final_nums,
    "excluded_features": {"leakage": config.LEAKAGE_FEATURES, "no_signal": prep.DROPPED_FEATURES,
                          "data_minimisation": ["age", "marital"], "replaced": {"pdays": "contacted_before"}},
    "row_cleaning": "src.preprocessing.clean_rows (dedupe, contacted_before, default no/not_no, merge illiterate, cap campaign at 10)",
    "operating_rules": {
        "main": {"name": "Capacity 20%", "how": "rank clients by score and call the top 20%",
                 "reference_threshold": round(THRESHOLDS["Capacity 20% (main)"], 4)},
        "alternative": {"name": "Max F1", "how": "call clients with score >= threshold",
                        "threshold": round(THRESHOLDS["Max F1 (alternative)"], 4)},
    },
    "test_performance": {k: round(float(v), 4) for k, v in overall.loc["Final RF (no age/marital)"].items()},
    "versions": {"python": sys.version.split()[0], "scikit-learn": sklearn.__version__, "pandas": pd.__version__},
}
(R / "final_model_config.json").write_text(json.dumps(final_config, indent=2, default=str))
overall.to_csv(R / "final_model_test_overall.csv")
ops.to_csv(R / "final_model_operating_rules.csv")
final_per_year.to_csv(R / "final_model_per_year.csv")
final_fair_age.to_csv(R / "final_model_fairness_age.csv")
final_fair_marital.to_csv(R / "final_model_fairness_marital.csv")
print("Saved models/final_model.joblib and results/final_model_*.csv / final_model_config.json")
print(json.dumps(final_config, indent=2, default=str))

Final model refitted on 41,176 clients in 9s
Saved models/final_model.joblib and results/final_model_*.csv / final_model_config.json
{
  "model": "RandomForestClassifier (scikit-learn Pipeline)",
  "created": "2026-10-10",
  "trained_on_rows": 41176,
  "dataset_sha256": "74adfc578bf77a7ff4bb1ba4a9f8709d9e3c6907342959c2c8416847e0afb4d8",
  "hyperparameters": {
    "class_weight": "balanced",
    "max_depth": 12,
    "max_features": 0.3,
    "min_samples_leaf": 10,
    "n_estimators": 300
  },
  "categorical_inputs": [
    "job",
    "education",
    "default",
    "contact",
    "month",
    "day_of_week",
    "poutcome"
  ],
  "numeric_inputs": [
    "campaign",
    "previous",
    "contacted_before",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed"
  ],
  "excluded_features": {
    "leakage": [
      "duration"
    ],
    "no_signal": [
      "housing",
      "loan"
    ],
    "data_minimisation": [
      "age",
      "marital"
    ],
 

## 7. Demo: scoring new clients into a ranked call list
How the bank would use the model before a campaign: take the client list, apply the same row cleaning, score, rank, and flag the top 20%.
Here 15 test clients stand in for a "new" list. For the top 3, SHAP shows the main reasons behind their score.

In [8]:
def rank_call_list(raw_clients, model, capacity=0.20):
    """Raw client records (same columns as the original file, without y) -> ranked call list."""
    clients = raw_clients.copy()
    clients["y"] = "no"                                    # placeholder so the shared cleaning function can run
    X = prep.clean_rows(clients).drop(columns=config.TARGET)
    out = X.assign(score=model.predict_proba(X)[:, 1]).sort_values("score", ascending=False)
    n_call = int(np.ceil(capacity * len(out)))
    out["priority"] = ["CALL" if i < n_call else "later" for i in range(len(out))]
    out.insert(0, "rank", np.arange(1, len(out) + 1))
    return out

# 15 raw client records (as the bank would have them before calling)
raw = pd.read_csv(config.RAW_FILE, sep=config.CSV_SEPARATOR).drop_duplicates().reset_index(drop=True)
demo_raw = raw.loc[Xte_pos.index[:15]].drop(columns="y")
call_list = rank_call_list(demo_raw, final_model)
call_list[["rank", "priority", "score", "job", "contact", "month", "poutcome", "campaign", "euribor3m"]].round(3)

,rank,priority,score,job,contact,month,poutcome,campaign,euribor3m
1,1,CALL,0.974,student,cellular,sep,success,2,0.880
3,2,CALL,0.495,blue-collar,cellular,may,nonexistent,2,1.266
10,3,CALL,0.494,entrepreneur,cellular,may,nonexistent,3,1.250
2,4,later,0.491,self-employed,cellular,may,failure,3,1.244
6,5,later,0.469,blue-collar,cellular,may,nonexistent,1,1.250
8,6,later,0.355,services,cellular,may,nonexistent,1,1.299
13,7,later,0.331,blue-collar,cellular,nov,nonexistent,1,4.021
4,8,later,0.294,technician,cellular,aug,nonexistent,2,4.964
11,9,later,0.288,blue-collar,telephone,jun,nonexistent,2,4.967
5,10,later,0.276,blue-collar,telephone,jun,nonexistent,4,4.959


In [9]:
prep_step, rf_step = final_model.named_steps["prep"], final_model.named_steps["model"]
top3 = call_list.head(3).drop(columns=["rank", "score", "priority"])
Xt = pd.DataFrame(prep_step.transform(top3), columns=prep_step.get_feature_names_out(), index=top3.index)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sv = shap.TreeExplainer(rf_step).shap_values(Xt)
sv = np.asarray(sv[1]) if isinstance(sv, list) else (np.asarray(sv)[..., 1] if np.ndim(sv) == 3 else np.asarray(sv))
for k, idx in enumerate(top3.index):
    contrib = pd.Series(sv[k], index=Xt.columns)
    # keep numeric inputs, and only the one-hot categories this client actually has
    keep = [c for c in Xt.columns if c in final_nums or Xt.loc[idx, c] == 1]
    contrib = contrib[keep].sort_values(key=abs, ascending=False).head(4)
    labels = [f"{c}={Xt.loc[idx, c]:g}" if c in final_nums else c for c in contrib.index]
    reasons = ", ".join(f"{lab} ({val:+.3f})" for lab, val in zip(labels, contrib.values))
    print(f"Rank {k + 1} (score {call_list.loc[idx, 'score']:.3f}): {reasons}")

Rank 1 (score 0.974): nr.employed=4963.6 (+0.147), euribor3m=0.88 (+0.098), poutcome_success (+0.089), emp.var.rate=-1.1 (+0.072)
Rank 2 (score 0.495): euribor3m=1.266 (+0.070), emp.var.rate=-1.8 (+0.055), nr.employed=5099.1 (-0.043), month_may (-0.041)
Rank 3 (score 0.494): euribor3m=1.25 (+0.057), emp.var.rate=-1.8 (+0.053), nr.employed=5099.1 (-0.048), month_may (-0.039)


## 8. Summary
Model card: `docs/model_card.md` · Recommendations and limitations: `docs/recommendations.md`.

**Final model (D27):** Random Forest (300 trees, max_depth 12, min_samples_leaf 10, max_features 0.3, balanced class weights), using
15 inputs: no `duration` (leakage), `housing`/`loan` (no signal), `age`/`marital` (data minimisation), with `pdays` replaced by `contacted_before`.
XGBoost is the documented fallback.

**Test-set performance (trained on the training set only):** PR-AUC **0.496**, ROC-AUC **0.814**, top-20% capture **67.0%**, the same as
the tuned RF with all features (0.4955), so removing `age` and `marital` cost nothing.

**Operating rules on the 8,236 test clients (928 subscribers)**
| Rule | Threshold | Calls | Subscribers found | Conversion | vs call everyone | Wasted calls |
|---|---|---|---|---|---|---|
| **Capacity 20% (main)** | 0.447 | 1,651 (20.0%) | **622 (67.0%)** | 37.7% | 3.3× | 1,029 |
| Max F1 (alternative) | 0.638 | 1,125 (13.7%) | 536 (57.8%) | **47.6%** | **4.2×** | 589 |
| Call everyone | – | 8,236 | 928 | 11.3% | 1× | 7,308 |

**Robustness and monitoring**
- Within-year ROC-AUC: 2008 **0.592**, 2009 **0.788**, 2010 **0.745**. Ranking is weakest when little contact history exists; regular retraining is needed.
- Although `age` is not an input, clients aged 61+ are still selected at 95–99% (proxies), with precision 42–56% (they genuinely respond more). Group outcomes must be monitored.

**Deployment package:** `models/final_model.joblib`, refitted on all 41,176 clients, plus `results/final_model_config.json`. The demo turns raw client records into a ranked call list with SHAP reasons. *(The 15 demo clients were part of the refit data; the demo illustrates usage, not performance.)*